In [10]:
import pandas as pd

# Load the dataset
df = pd.read_csv("survey_results_public.csv")

# Select the required column
age_data = df["Age1stCode"]

# Convert the column into numeric values
age_data = pd.to_numeric(age_data, errors="coerce")

# Calculate average age
average_age = age_data.mean()

print("Average age of developers when they first coded:",
      round(average_age, 2))

Average age of developers when they first coded: 15.46


In [12]:
data = df[["Country", "LanguageWorkedWith"]].copy()

# 3. Remove rows with missing country or language
data = data.dropna(
    subset=["Country", "LanguageWorkedWith"]
)

# 4. Identify respondents who worked with Python
data["KnowsPython"] = data["LanguageWorkedWith"].apply(
    lambda languages: "Python" in languages.split(";")
)

# 5. Group by country and calculate percentage
result = (
    data.groupby("Country")
    .agg(
        TotalDevelopers=("KnowsPython", "size"),
        PythonDevelopers=("KnowsPython", "sum")
    )
    .reset_index()
)

# 6. Calculate percentage
result["PythonPercentage"] = (
    result["PythonDevelopers"]
    / result["TotalDevelopers"]
    * 100
).round(2)



# 8. Display result
print(result.to_string(index=False))

                                  Country  TotalDevelopers  PythonDevelopers  PythonPercentage
                              Afghanistan               39                 8             20.51
                                  Albania               83                23             27.71
                                  Algeria              126                40             31.75
                                  Andorra                7                 0              0.00
                                   Angola                4                 2             50.00
                      Antigua and Barbuda                7                 0              0.00
                                Argentina              547               191             34.92
                                  Armenia               84                28             33.33
                                Australia             1880               790             42.02
                                  Austria         

In [13]:
import pycountry_convert as pc



# 2. Select relevant columns
data = df[["Country", "ConvertedComp"]].copy()

# 3. Clean salary
data["ConvertedComp"] = pd.to_numeric(
    data["ConvertedComp"], errors="coerce"
)

# Compensation must be positive
data = data.dropna(subset=["Country", "ConvertedComp"])
data = data[data["ConvertedComp"] > 0].copy()

# 4. Country to continent mapping
def get_continent(country):
    try:
        country_code = pc.country_name_to_country_alpha2(country)
        continent_code = pc.country_alpha2_to_continent_code(
            country_code
        )

        continent_mapping = {
            "AF": "Africa",
            "AS": "Asia",
            "EU": "Europe",
            "NA": "North America",
            "SA": "South America",
            "OC": "Oceania"
        }

        return continent_mapping.get(continent_code)

    except (KeyError, TypeError, ValueError):
        return None

# 5. Apply mapping
data["Continent"] = data["Country"].apply(get_continent)

# 6. Inspect unmapped countries
unmapped = data.loc[
    data["Continent"].isna(), "Country"
].drop_duplicates()

print("Unmapped countries:")
print(unmapped.tolist())

# 7. Keep mapped records
data = data.dropna(subset=["Continent"])

# 8. Aggregate salary by continent
result = (
    data.groupby("Continent")
    .agg(
        DeveloperCount=("ConvertedComp", "count"),
        AverageSalaryUSD=("ConvertedComp", "mean")
    )
    .reset_index()
)

# 9. Format output
result["AverageSalaryUSD"] = (
    result["AverageSalaryUSD"].round(2)
)

result = result.sort_values(
    "AverageSalaryUSD", ascending=False
)

print(result.to_string(index=False))

Unmapped countries:
['Hong Kong (S.A.R.)', 'Venezuela, Bolivarian Republic of...', 'Other Country (Not Listed Above)', 'The former Yugoslav Republic of Macedonia', 'Libyan Arab Jamahiriya', 'Republic of Korea', 'Congo, Republic of the...', 'Timor-Leste']
    Continent  DeveloperCount  AverageSalaryUSD
North America           17965         226232.51
      Oceania            1603         167263.40
       Europe           23681          97296.63
         Asia            8435          36857.04
       Africa            1242          35150.51
South America            2434          33105.76


In [14]:
languages = df["LanguageDesireNextYear"].dropna()

# 3. Split multiple language responses
language_lists = languages.str.split(";")

# 4. Convert lists into individual rows
all_languages = language_lists.explode()

# 5. Clean whitespace
all_languages = all_languages.str.strip()

# 6. Remove empty values
all_languages = all_languages[all_languages != ""]

# 7. Count language preferences
language_counts = all_languages.value_counts()

# 8. Create report
result = language_counts.reset_index()
result.columns = ["ProgrammingLanguage", "DesiredCount"]

# 9. Calculate percentages
total_responses = languages.shape[0]

result["DesiredPercentage"] = (
    result["DesiredCount"] / total_responses * 100
).round(2)

# 10. Sort by popularity
result = result.sort_values(
    "DesiredCount",
    ascending=False
)

print(result.head(10).to_string(index=False))

# 11. Export report
result.to_csv("desired_languages_2020.csv", index=False)

# 12. Identify most desired language
print("\nMost desired language:")
print(result.iloc[0])

  ProgrammingLanguage  DesiredCount  DesiredPercentage
           JavaScript         44739              53.20
               Python         40006              47.58
             HTML/CSS         36645              43.58
                  SQL         33566              39.92
           TypeScript         23720              28.21
                 Java         23508              27.96
                   C#         22449              26.70
Bash/Shell/PowerShell         21303              25.33
                   Go         17060              20.29
                  C++         16856              20.05

Most desired language:
ProgrammingLanguage    JavaScript
DesiredCount                44739
DesiredPercentage            53.2
Name: 0, dtype: object


In [15]:
data = df[["Country", "Hobbyist"]].copy()

# 3. Clean missing values
data = data.dropna(subset=["Country", "Hobbyist"])

# 4. Map country to continent
def get_continent(country):
    try:
        country_code = pc.country_name_to_country_alpha2(country)
        continent_code = pc.country_alpha2_to_continent_code(
            country_code
        )

        mapping = {
            "AF": "Africa",
            "AS": "Asia",
            "EU": "Europe",
            "NA": "North America",
            "SA": "South America",
            "OC": "Oceania"
        }

        return mapping.get(continent_code)

    except (KeyError, TypeError, ValueError):
        return None

data["Continent"] = data["Country"].apply(get_continent)

# 5. Inspect unmapped countries
unmapped = data.loc[
    data["Continent"].isna(), "Country"
].drop_duplicates()

print("Unmapped countries:")
print(unmapped.tolist())

# 6. Keep mapped records
data = data.dropna(subset=["Continent"])

# 7. Normalize hobby responses
data["Hobbyist"] = data["Hobbyist"].str.strip()

# 8. Keep valid Yes/No responses
data = data[data["Hobbyist"].isin(["Yes", "No"])]

# 9. Create boolean indicator
data["IsHobbyist"] = data["Hobbyist"].eq("Yes")

# 10. Aggregate by continent
result = (
    data.groupby("Continent")
    .agg(
        TotalRespondents=("IsHobbyist", "size"),
        HobbyistCount=("IsHobbyist", "sum")
    )
    .reset_index()
)

# 11. Calculate percentage
result["HobbyistPercentage"] = (
    result["HobbyistCount"]
    / result["TotalRespondents"]
    * 100
).round(2)

# 12. Sort report
result = result.sort_values(
    "Continent"
)

print(result.to_string(index=False))

Unmapped countries:
['Hong Kong (S.A.R.)', 'Libyan Arab Jamahiriya', 'Venezuela, Bolivarian Republic of...', 'Other Country (Not Listed Above)', 'The former Yugoslav Republic of Macedonia', 'Republic of Korea', 'Congo, Republic of the...', 'Timor-Leste']
    Continent  TotalRespondents  HobbyistCount  HobbyistPercentage
       Africa              2732           2149               78.66
         Asia             18345          14314               78.03
       Europe             35789          29362               82.04
North America             25494          20410               80.06
      Oceania              2434           2013               82.70
South America              3434           2598               75.66


In [21]:
data = df[["Country", "JobSat", "CareerSat"]].copy()

# Map countries to continents
data["Continent"] = data["Country"].apply(get_continent)

# Remove unmapped countries
data = data.dropna(subset=["Continent"]).copy()

# Treat blank strings as missing
for col in ["JobSat", "CareerSat"]:
    data[col] = data[col].astype("string").str.strip()
    data[col] = data[col].replace("", pd.NA)

# Aggregate by continent
result = (
    data.groupby("Continent")
    .agg(
        JobSatisfactionRespondents=("JobSat", "count"),
        CareerSatisfactionRespondents=("CareerSat", "count")
    )
    .reset_index()
)

print(result.to_string(index=False))

# Export report
result.to_csv(
    "satisfaction_respondents_by_continent.csv",
    index=False
)

    Continent  JobSatisfactionRespondents  CareerSatisfactionRespondents
       Africa                        1696                           1854
         Asia                       13497                          14042
       Europe                       29307                          29804
North America                       21220                          21732
      Oceania                        2027                           2072
South America                        2862                           2951
